# Hierarchical clustering and DBSCAN

k-means needs $k$ up front and assumes round clusters. This notebook covers two methods that relax those assumptions. **Hierarchical clustering** builds a tree of merges and lets you choose the number of clusters afterwards by cutting the tree. **DBSCAN** defines a cluster as a dense region, so it finds clusters of any shape, needs no $k$, and labels sparse points as noise.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from common.datasets import load_wine_data, make_blobs_data, make_circles_data, make_moons_data
from common.plotting import plot_clusters

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

**Hierarchical (agglomerative).** Start with every point as its own cluster. Find the two closest clusters, merge them, and repeat until one cluster holds everything. The record of merges is a tree called a **dendrogram**. Cut the tree at a height and you get a set of clusters. A low cut gives many small clusters, a high cut gives a few big ones, so one run answers every value of $k$.

**DBSCAN.** Stand on a point and look around with a circle of radius $\varepsilon$ (`eps`). If the circle holds enough neighbours, you are in a crowded place. Clusters are crowds joined by overlapping circles. Points that are alone in the dark are noise. Think of towns on a map: a town is a dense patch of houses, and a few farms in between belong to no town.

## 2. Maths and cost function

**Linkage.** The distance between two points is clear. The distance between two *clusters* is a choice, called the linkage, and it decides the shape of the result.

| Linkage | Distance between clusters $A$ and $B$ | Tends to produce |
|---|---|---|
| single | smallest pairwise distance | long chains, follows shapes, sensitive to noise bridges |
| complete | largest pairwise distance | compact clusters of similar diameter |
| average | mean of all pairwise distances | a compromise |
| ward | increase in total within-cluster variance if merged | round, similar-sized clusters, like k-means |

Ward is the same objective as k-means (sum of squared distances to the centre) applied greedily, which is why it behaves like k-means and is the usual default for blob-like data.

There is no global cost function for hierarchical clustering. Each merge is the locally best one and is never undone, so the method is greedy.

**DBSCAN** has two parameters and three kinds of point:

- A **core** point has at least `min_samples` points (itself included) within distance $\varepsilon$.
- A **border** point is within $\varepsilon$ of a core point but is not core itself.
- Everything else is **noise**, label $-1$.

Two core points within $\varepsilon$ of each other belong to the same cluster, and border points join the cluster of a core point near them. Clusters grow by chaining through core points, which is why shape does not matter. DBSCAN also has no cost function. The result is fixed by the data and the two parameters.

## 3. From scratch

**Agglomerative, single linkage.** Twelve points in two tight groups. We compute the merge order by repeatedly joining the two closest clusters, then compare with scipy.

In [ ]:
pts = np.vstack([rng.normal([0, 0], 0.4, (6, 2)), rng.normal([3, 3], 0.4, (6, 2))])
D = squareform(pdist(pts))
clusters = {i: [i] for i in range(len(pts))}
merges = []
while len(clusters) > 1:
    keys = list(clusters)
    best = min(
        (
            (D[np.ix_(clusters[a], clusters[b])].min(), a, b)
            for i, a in enumerate(keys)
            for b in keys[i + 1 :]
        ),
    )
    d, a, b = best
    merges.append(round(float(d), 3))
    clusters[a] = clusters[a] + clusters.pop(b)
Z = linkage(pts, method="single")
print("merge heights, mine: ", np.array(merges))
print("merge heights, scipy:", Z[:, 2].round(3))

The heights match. The last merge, at a height near 3, is the long jump between the two groups. That jump is the signal: cutting the tree just below it gives two clusters.

**DBSCAN.** The algorithm is short enough to write in a few lines. Find each point's neighbours, mark core points, then grow clusters by a breadth-first search through core points.

In [ ]:
def dbscan(X: np.ndarray, eps: float, min_samples: int) -> np.ndarray:
    dist = squareform(pdist(X))
    neighbours = [np.flatnonzero(row <= eps) for row in dist]  # includes the point itself
    core = np.array([len(nb) >= min_samples for nb in neighbours])
    labels = np.full(len(X), -1)
    cluster = 0
    for start in np.flatnonzero(core):
        if labels[start] != -1:
            continue
        labels[start] = cluster
        queue = [start]
        while queue:
            for nb in neighbours[queue.pop()]:
                if labels[nb] == -1:
                    labels[nb] = cluster  # core or border point joins the cluster
                    if core[nb]:
                        queue.append(nb)  # only core points keep the chain going
        cluster += 1
    return labels


Xm_df, ym = make_moons_data(300, noise=0.08, seed=SEED)
Xm = Xm_df.to_numpy()
mine = dbscan(Xm, eps=0.2, min_samples=5)
ref = DBSCAN(eps=0.2, min_samples=5).fit(Xm).labels_
print(
    "clusters:", sorted(int(c) for c in set(mine) - {-1}), " noise points:", int((mine == -1).sum())
)
print("same as sklearn (up to cluster numbering):", adjusted_rand_score(mine, ref) == 1.0)

## 4. Library version and the cases k-means fails

Run k-means, Ward, single-linkage and DBSCAN on moons and on circles, and score each against the true shapes with ARI (1 is perfect, near 0 is chance).

In [ ]:
Xr_df, yr = make_circles_data(300, noise=0.05, seed=SEED)
sets = {"moons": (Xm, ym, 0.2), "circles": (Xr_df.to_numpy(), yr, 0.15)}
models = {
    "k-means": lambda e: KMeans(2, n_init=10, random_state=SEED),
    "ward": lambda e: AgglomerativeClustering(2, linkage="ward"),
    "single": lambda e: AgglomerativeClustering(2, linkage="single"),
    "DBSCAN": lambda e: DBSCAN(eps=e, min_samples=5),
}
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for row, (name, (Xs, ys, eps)) in zip(axes, sets.items()):
    for ax, (mname, make) in zip(row, models.items()):
        lab = make(eps).fit_predict(Xs)
        plot_clusters(Xs, lab, ax, title=f"{name}, {mname}: ARI {adjusted_rand_score(ys, lab):.2f}")
plt.tight_layout()
plt.show()

Single linkage and DBSCAN recover both shapes (ARI near 1). k-means and Ward both cut straight through them, because their objective rewards compact blobs. Single linkage works here because the shapes are connected chains of close points. That same property is a weakness when noise forms a bridge between two groups (section 6).

## 5. Tuning

**Hierarchical: read the dendrogram.** The vertical axis is the distance at which clusters merged. Long vertical lines before a merge mean the two clusters were far apart, so cut in a gap. Use the wine data (178 wines, 13 chemical measurements) which has three cultivars. Scale first, because the features are on different scales.

In [ ]:
Xw, yw = load_wine_data()
Xw_s = StandardScaler().fit_transform(Xw)
Zw = linkage(Xw_s, method="ward")
fig, ax = plt.subplots(figsize=(10, 3.5))
dendrogram(Zw, truncate_mode="lastp", p=20, no_labels=True, ax=ax, color_threshold=0)
ax.axhline(20, color="r", ls="--")
ax.set_ylabel("merge distance (Ward)")
plt.show()
print("last five merge heights:", Zw[-5:, 2].round(1))
for k in (2, 3, 4):
    lab = fcluster(Zw, k, criterion="maxclust")
    print(f"cut into {k}: ARI vs cultivar {adjusted_rand_score(yw, lab):.3f}")

The last two merges (heights 27.7 and 35.4) sit well above the third from last (12.6), which points to three clusters. The red line cuts there, and the cut into three agrees reasonably with the true cultivars (ARI 0.79; the cultivars overlap in chemistry, so a perfect score is not expected). The choice of linkage matters, so look at several.

In [ ]:
for method in ("ward", "complete", "average", "single"):
    lab = fcluster(linkage(Xw_s, method=method), 3, criterion="maxclust")
    print(f"{method:>8}: ARI {adjusted_rand_score(yw, lab):.3f}  sizes {np.bincount(lab)[1:]}")

Single and average linkage return one giant cluster of 174 points and two stragglers of 3 and 1: a few points sit away from the main body and get cut off alone. Ward gives balanced groups of 56, 64 and 58 and the best ARI.

**DBSCAN: min_samples and eps.** Start with `min_samples`. A common rule is $\text{min\_samples}\ge d+1$, and about $2d$ for larger or noisier data, where $d$ is the number of features. For two features, 4 to 5 is usual. Larger values need denser regions to count as clusters, so more points become noise.

Then `eps`. The **k-distance plot** helps: for each point find the distance to its `min_samples`-th nearest neighbour, then sort them. Points inside clusters have small distances and noise points have large ones, so the sorted curve has a knee. Set `eps` near the knee.

In [ ]:
nn = NearestNeighbors(n_neighbors=5).fit(Xm)
kdist = np.sort(nn.kneighbors(Xm)[0][:, -1])
fig, ax = plt.subplots()
ax.plot(kdist)
ax.axhline(0.2, color="r", ls="--")
ax.set(
    xlabel="points sorted by distance",
    ylabel="distance to 5th nearest neighbour",
    title="k-distance plot (moons)",
)
plt.show()
rows = []
for eps in (0.05, 0.1, 0.2, 0.4, 0.8):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xm)
    rows.append(
        (eps, len(set(lab) - {-1}), int((lab == -1).sum()), round(adjusted_rand_score(ym, lab), 2))
    )
print("eps, clusters, noise points, ARI")
for r in rows:
    print(r)
print(
    "k-distance values at the 50th, 90th, 99th percentile:",
    np.percentile(kdist, [50, 90, 99]).round(2),
)

The knee sits around 0.15 to 0.25. Too small an `eps` shatters the data: 0.05 gives 4 clusters and 279 noise points out of 300, and 0.1 gives 19 clusters. From 0.4 up, both moons join into one cluster. Only 0.2 gets it right. The usable range is fairly narrow, which is the main practical cost of DBSCAN.

## 6. What breaks

**Noise bridges break single linkage.** Add a thin line of points between two blobs and the chain connects them.

In [ ]:
Xb_df, yb = make_blobs_data(200, centers=2, cluster_std=0.6, seed=7)
Xb = Xb_df.to_numpy()
a, b = Xb[yb == 0].mean(axis=0), Xb[yb == 1].mean(axis=0)
bridge = a + np.linspace(0, 1, 15)[:, None] * (b - a)
Xbr = np.vstack([Xb, bridge])
ybr = np.r_[yb, np.full(len(bridge), 2)]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharex=True, sharey=True)
for ax, (name, model) in zip(
    axes,
    [
        ("ward", AgglomerativeClustering(2, linkage="ward")),
        ("single", AgglomerativeClustering(2, linkage="single")),
    ],
):
    lab = model.fit_predict(Xbr)
    print(
        name,
        "cluster sizes:",
        np.bincount(lab),
        " ARI vs the two blobs (bridge excluded):",
        round(adjusted_rand_score(yb, lab[: len(yb)]), 2),
    )
    plot_clusters(Xbr, lab, ax, title=name)
db = DBSCAN(eps=0.3, min_samples=5).fit_predict(Xbr)
print(
    "DBSCAN clusters:",
    sorted(int(c) for c in set(db) - {-1}),
    " noise points:",
    int((db == -1).sum()),
)
plot_clusters(
    Xbr, db, axes[2], title=f"DBSCAN eps=0.3, min_samples=5: {len(set(db) - {-1})} clusters"
)
plt.show()

Ward is not bothered by the bridge: it recovers both blobs (ARI 1.0) and shares the bridge points between them. Single linkage follows the chain of close points and ends up with one cluster of 214 points and a single leftover, so the two blobs are merged. DBSCAN at `eps=0.3` keeps two clusters and marks 41 points as noise, most of them the bridge. That works only because the bridge is sparse. Make it dense enough to hold core points and DBSCAN chains along it like single linkage does.

**DBSCAN with different densities.** One `eps` cannot suit a tight cluster and a loose one.

In [ ]:
tight = rng.normal([0, 0], 0.3, (150, 2))
loose = rng.normal([6, 0], 1.5, (150, 2))
Xd = np.vstack([tight, loose])
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharex=True, sharey=True)
for ax, eps in zip(axes, (0.25, 0.6, 1.2)):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xd)
    print(f"eps={eps}: noise points", int((lab == -1).sum()), " clusters", len(set(lab) - {-1}))
    plot_clusters(
        Xd,
        lab,
        ax,
        title=f"eps={eps}: {len(set(lab) - {-1})} clusters, {int((lab == -1).sum())} noise",
    )
plt.show()

At `eps=0.25` the tight cluster is found but 136 of the 300 points, mostly from the loose cluster, are noise. At 0.6 the noise falls to 30 points. At 1.2 almost nothing is noise, but the two clusters have merged into one. No single value serves both densities. `HDBSCAN` (in scikit-learn as `sklearn.cluster.HDBSCAN`) removes the single `eps` by searching across scales, and is worth trying when densities differ.

**Scale and dimension.** Both methods use distances, so scale the features first, exactly as with k-means. In many dimensions distances flatten out (see `03_classical_ml/21_knn.ipynb`), so `eps` becomes hard to set and clusters vanish into noise. Reduce dimensions first (notebook 04).

**Cost.** Agglomerative clustering stores all pairwise distances, so memory is $O(n^2)$ and it is impractical beyond roughly 10,000 to 20,000 points. DBSCAN with a spatial index runs in about $O(n\log n)$ in low dimensions.

| Need | Reach for |
|---|---|
| Round blobs, known $k$, big data | k-means |
| A tree you can cut at several levels, small data | Ward or average linkage |
| Odd shapes, noise labelled, no $k$ | DBSCAN |
| Odd shapes and varying density | HDBSCAN |

Next, a model that gives each point a probability of belonging to each cluster.